In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

df = pd.read_csv("uber.csv")

print(df.shape)
df.head()

In [ ]:
df.dropna(inplace=True)
df.drop_duplicates(inplace=True)

dt = pd.to_datetime(df['pickup_datetime'], errors='coerce')
df = df[dt.notna()].copy()
dt = pd.to_datetime(df['pickup_datetime'])

df['hour'] = dt.dt.hour
df['day'] = dt.dt.day
df['month'] = dt.dt.month
df['weekday'] = dt.dt.weekday

df = df[
    df['pickup_latitude'].between(-90, 90) &
    df['dropoff_latitude'].between(-90, 90) &
    df['pickup_longitude'].between(-180, 180) &
    df['dropoff_longitude'].between(-180, 180)
].copy()

lat1, lon1, lat2, lon2 = np.radians([
    df['pickup_latitude'], df['pickup_longitude'],
    df['dropoff_latitude'], df['dropoff_longitude']
])

df['distance_km'] = 6371 * 2 * np.arcsin(np.sqrt(
    np.sin((lat2-lat1)/2)**2 +
    np.cos(lat1)*np.cos(lat2)*np.sin((lon2-lon1)/2)**2
))

df.drop(columns=['key', 'pickup_datetime', 'Unnamed: 0',
                 'pickup_latitude', 'pickup_longitude',
                 'dropoff_latitude', 'dropoff_longitude'],
        inplace=True, errors='ignore')

df = df[
    (df['fare_amount'].between(0.01, 100)) &
    (df['distance_km'].between(0.01, 100)) &
    (df['passenger_count'].between(1, 6))
]

df.head()

In [ ]:
for col in ['fare_amount', 'distance_km']:
    q1, q3 = df[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    count = ((df[col] < q1 - 1.5*iqr) |
             (df[col] > q3 + 1.5*iqr)).sum()
    print(col, "Outliers:", count)

sns.boxplot(data=df[['fare_amount', 'distance_km']])
plt.show()

In [ ]:
corr = df.corr(numeric_only=True)

sns.heatmap(corr, annot=True, cmap='coolwarm')
plt.show()

In [6]:
X = df.drop('fare_amount', axis=1)
y = df['fare_amount']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

lr = LinearRegression()
lr.fit(X_train, y_train)

lr_pred = lr.predict(X_test)

In [7]:
rf = RandomForestRegressor(
    n_estimators=50, max_depth=15,
    random_state=42, n_jobs=-1
)

rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)

In [ ]:
results = pd.DataFrame({
    'Model': ['Linear Regression', 'Random Forest'],
    'R2 Score': [r2_score(y_test, lr_pred),
                 r2_score(y_test, rf_pred)],
    'MSE': [mean_squared_error(y_test, lr_pred),
            mean_squared_error(y_test, rf_pred)],
    'RMSE': [np.sqrt(mean_squared_error(y_test, lr_pred)),
             np.sqrt(mean_squared_error(y_test, rf_pred))],
    'MAE': [mean_absolute_error(y_test, lr_pred),
            mean_absolute_error(y_test, rf_pred)]
})

display(results)